# Task Model and  Feasibility Layer

In [1]:
import sys, os
sys.path.append(os.path.abspath("C:/Users/bomma/Desktop/DESKTOP_MAIN_DISP/MATHS_PROJECT_SEM5/A_project_re/project_COMPLETE_all_5_days/project/src"))
from robots import RobotFleet
from tasks import TaskManager, feasible_robots_for_task

DATASET_ROOT =  "C:/Users/bomma/Desktop/DESKTOP_MAIN_DISP/MATHS_PROJECT_SEM5/A_project_re/project_COMPLETE_all_5_days/project/data/dataset"
fleet = RobotFleet(DATASET_ROOT)
tm = TaskManager(DATASET_ROOT)
print(tm.summary())
tm.as_dataframe()


Tasks: 30 -> {'MEDICAL': 2, 'SEARCH': 6, 'INSPECTION': 5, 'RESCUE': 4, 'MONITORING': 3, 'TRANSPORT': 4, 'SURVEILLANCE': 6}, emergencies=3


,task_id,task_type,priority,priority_label,emergency,status,assigned_robot
0,T01,MEDICAL,2,Normal,False,PENDING,None
1,T02,SEARCH,3,High,False,PENDING,None
2,T03,INSPECTION,5,Emergency,True,PENDING,None
3,T04,INSPECTION,3,High,False,PENDING,None
4,T05,RESCUE,3,High,False,PENDING,None
5,T06,MONITORING,2,Normal,False,PENDING,None
6,T07,SEARCH,3,High,False,PENDING,None
7,T08,RESCUE,2,Normal,False,PENDING,None
8,T09,RESCUE,3,High,False,PENDING,None
9,T10,MEDICAL,3,High,False,PENDING,None


## Composition check 

In [2]:
assert len(tm.tasks) == 30
n_emergency = sum(1 for t in tm.tasks.values() if t.emergency)
print(f"30 tasks confirmed. Emergency tasks: {n_emergency}")


30 tasks confirmed. Emergency tasks: 3


## Feasibility layer: multi-type tasks vs single-type tasks

In [3]:
multi_type_tasks = [t for t in tm.tasks.values() if len(t.allowed_robot_types) > 1]
print(f"{len(multi_type_tasks)} tasks allow more than one robot type, e.g.:")
for t in multi_type_tasks[:5]:
    feasible = feasible_robots_for_task(list(fleet.robots.values()), t)
    print(f"  {t.task_id} ({t.task_type}) allowed={t.allowed_robot_types} "
          f"-> {len(feasible)} feasible robots: {[r.robot_id for r in feasible]}")


3 tasks allow more than one robot type, e.g.:
  T03 (INSPECTION) allowed={'UGV', 'UAV'} -> 8 feasible robots: ['UGV01', 'UGV02', 'UGV03', 'UGV04', 'UAV01', 'UAV02', 'UAV03', 'UAV04']
  T13 (TRANSPORT) allowed={'USV', 'UAV'} -> 4 feasible robots: ['USV01', 'USV02', 'USV03', 'USV04']
  T27 (SURVEILLANCE) allowed={'USV', 'UAV'} -> 5 feasible robots: ['USV01', 'USV02', 'USV03', 'USV04', 'UAV03']


In [4]:
# An infeasible edge must never appear: pick a USV-only task and confirm no UGV/UAV qualifies
water_only = [t for t in tm.tasks.values() if t.allowed_robot_types == {"USV"}]
if water_only:
    t = water_only[0]
    feasible = feasible_robots_for_task(list(fleet.robots.values()), t)
    assert all(r.robot_type == "USV" for r in feasible)
    print(f"OK: {t.task_id} (USV-only) correctly excludes UGV/UAV -> {[r.robot_id for r in feasible]}")
else:
    print("No purely USV-only task in this dataset instance; skipping this specific check.")


OK: T11 (USV-only) correctly excludes UGV/UAV -> ['USV01', 'USV02', 'USV03', 'USV04']
